# Merging Census Race Data into SDPD Law Beats (neighborhoods)

In [37]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [38]:
# Load Census tract and raceshapefile
tracts_race = gpd.read_file("C:\\UC San Diego\\Capstone\\tl_2025_06_tract\\tl_2025_06_tract.shp")

# Load SDPD Beats shapefile
neighborhoods = gpd.read_file("C:\\UC San Diego\\Capstone\\SDPD_Beats_shapefile\\SDPD_Beats.shp")


In [39]:
# Reproject to a local coordinate system
tracts_race = tracts_race.to_crs(epsg=2230)
neighborhoods = neighborhoods.to_crs(epsg=2230)

In [40]:
# Calculate area of each neighborhood
neighborhoods["neighborhood_area"] = neighborhoods.geometry.area

In [41]:
neighborhoods.head()

,NAME,Shape_Leng,Shape_Area,geometry,neighborhood_area
0,NORTH CITY,78873.790699,1.331479e+08,"POLYGON ((6258473.516 1939877.994, 6258489.997...",1.331479e+08
1,NESTOR,35035.587421,4.154836e+07,"POLYGON ((6302781 1793246.001, 6302905 1793244...",4.154836e+07
2,BIRDLAND,25427.283539,2.168501e+07,"POLYGON ((6284667.652 1874418.895, 6284694.392...",2.168501e+07
3,CHEROKEE POINT,13323.824075,9.452499e+06,"POLYGON ((6297431.051 1853639.085, 6297418.091...",9.452499e+06
4,LOMA PORTAL,27729.191262,1.889037e+07,"POLYGON ((6263446.993 1854500.089, 6263451.987...",1.889037e+07


In [42]:
tracts_race.head()

,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,...,P9_066N,P9_067N,P9_068N,P9_069N,P9_070N,P9_071N,P9_072N,P9_073N,Tract_Area,geometry
0,06,073,017039,06073017039,1400000US06073017039,170.39,Census Tract 170.39,G5020,S,4787636,...,0,0,0,0,0,0,0,0,5.158754e+07,"POLYGON ((6299072.692 1925846.218, 6299296.436..."
1,06,073,017040,06073017040,1400000US06073017040,170.40,Census Tract 170.40,G5020,S,2952353,...,0,0,0,0,0,0,0,0,3.177713e+07,"POLYGON ((6312422.551 1927234.134, 6312424.237..."
2,06,073,017043,06073017043,1400000US06073017043,170.43,Census Tract 170.43,G5020,S,3035943,...,1,0,0,0,0,0,0,0,3.267714e+07,"POLYGON ((6295855.052 1915019.22, 6295923.62 1..."
3,06,073,003103,06073003103,1400000US06073003103,31.03,Census Tract 31.03,G5020,S,2039646,...,0,0,0,0,0,0,0,0,2.195509e+07,"POLYGON ((6313929.954 1837465.67, 6313933.415 ..."
4,06,073,003105,06073003105,1400000US06073003105,31.05,Census Tract 31.05,G5020,S,1210504,...,0,0,0,0,0,0,0,0,1.303005e+07,"POLYGON ((6319459.883 1839902.029, 6319618.61 ..."


In [43]:
tracts_race.columns

Index(['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME',
       'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT',
       'INTPTLON', 'GEO_ID', 'NAME_1', 'P9_001N', 'P9_002N', 'P9_003N',
       'P9_004N', 'P9_005N', 'P9_006N', 'P9_007N', 'P9_008N', 'P9_009N',
       'P9_010N', 'P9_011N', 'P9_012N', 'P9_013N', 'P9_014N', 'P9_015N',
       'P9_016N', 'P9_017N', 'P9_018N', 'P9_019N', 'P9_020N', 'P9_021N',
       'P9_022N', 'P9_023N', 'P9_024N', 'P9_025N', 'P9_026N', 'P9_027N',
       'P9_028N', 'P9_029N', 'P9_030N', 'P9_031N', 'P9_032N', 'P9_033N',
       'P9_034N', 'P9_035N', 'P9_036N', 'P9_037N', 'P9_038N', 'P9_039N',
       'P9_040N', 'P9_041N', 'P9_042N', 'P9_043N', 'P9_044N', 'P9_045N',
       'P9_046N', 'P9_047N', 'P9_048N', 'P9_049N', 'P9_050N', 'P9_051N',
       'P9_052N', 'P9_053N', 'P9_054N', 'P9_055N', 'P9_056N', 'P9_057N',
       'P9_058N', 'P9_059N', 'P9_060N', 'P9_061N', 'P9_062N', 'P9_063N',
       'P9_064N', 'P9_065N', 'P9_066N', 'P9_067N', 'P9_068

In [50]:
tracts_race = tracts_race.rename(columns={
    'P9_001N': 'Total_Pop', 
    'P9_002N': 'Hispanic_Latino', 
    'P9_005N': 'White', 
    'P9_006N': 'Black_African_American', 
    'P9_007N': 'American_Indian_Alaska_Native', 
    'P9_008N': 'Asian', 
    'P9_009N': 'Native_Hawaiian_Pacific_Islander',
    'P9_010N': 'Other_Race'})

In [51]:
tracts_race.columns

Index(['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME',
       'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT',
       'INTPTLON', 'GEO_ID', 'NAME_1', 'Total_Pop', 'Hispanic_Latino',
       'P9_003N', 'P9_004N', 'White', 'Black_African_American',
       'American_Indian_Alaska_Native', 'Asian',
       'Native_Hawaiian_Pacific_Islander', 'Other_Race', 'P9_011N', 'P9_012N',
       'P9_013N', 'P9_014N', 'P9_015N', 'P9_016N', 'P9_017N', 'P9_018N',
       'P9_019N', 'P9_020N', 'P9_021N', 'P9_022N', 'P9_023N', 'P9_024N',
       'P9_025N', 'P9_026N', 'P9_027N', 'P9_028N', 'P9_029N', 'P9_030N',
       'P9_031N', 'P9_032N', 'P9_033N', 'P9_034N', 'P9_035N', 'P9_036N',
       'P9_037N', 'P9_038N', 'P9_039N', 'P9_040N', 'P9_041N', 'P9_042N',
       'P9_043N', 'P9_044N', 'P9_045N', 'P9_046N', 'P9_047N', 'P9_048N',
       'P9_049N', 'P9_050N', 'P9_051N', 'P9_052N', 'P9_053N', 'P9_054N',
       'P9_055N', 'P9_056N', 'P9_057N', 'P9_058N', 'P9_059N', 'P9_060N',
       'P9_06

In [52]:
race_cols = [
    "Hispanic_Latino",
    "White",
    "Black_African_American",
    "American_Indian_Alaska_Native",
    "Asian",
    "Native_Hawaiian_Pacific_Islander",
    "Other_Race"
]

In [53]:
# Perform spatial intersection
intersections = gpd.overlay(
    tracts_race[["GEOIDFQ", "Tract_Area","Total_Pop", *race_cols, "geometry"]],
    neighborhoods[['NAME', "geometry", "neighborhood_area"]],
    how="intersection"
)

In [54]:
# Calculate area of intersection and weight
# Area weight: share of each tract inside each beat
intersections["intersect_area"] = intersections.geometry.area
intersections["area_weight"] = intersections["intersect_area"] / intersections["Tract_Area"]

In [71]:
# Estimate population and race counts inside each beat
# For each column, multiply by area weight to get the estimated count in each beat
for col in ["Total_Pop", *race_cols]:
    intersections[f"{col}_weighted"] = intersections[col] * intersections["area_weight"]

weighted_cols = [f"{col}_weighted" for col in ["Total_Pop", *race_cols]]

In [72]:
intersections.head()

,GEOIDFQ,Tract_Area,Total_Pop,Hispanic_Latino,White,Black_African_American,American_Indian_Alaska_Native,Asian,Native_Hawaiian_Pacific_Islander,Other_Race,...,intersect_area,area_weight,Total_Pop_weighted,Hispanic_Latino_weighted,White_weighted,Black_African_American_weighted,American_Indian_Alaska_Native_weighted,Asian_weighted,Native_Hawaiian_Pacific_Islander_weighted,Other_Race_weighted
0,1400000US06073017039,5.158754e+07,6915,814,3084,164,2,2360,26,48,...,2.088671e+04,0.000405,2.799738,0.329571,1.248647,0.066400,0.000810,0.955514,0.010527,0.019434
1,1400000US06073017039,5.158754e+07,6915,814,3084,164,2,2360,26,48,...,3.762349e+07,0.729314,5043.204025,593.661327,2249.203357,119.607442,1.458627,1721.180260,18.962155,35.007056
2,1400000US06073017039,5.158754e+07,6915,814,3084,164,2,2360,26,48,...,1.302679e+07,0.252518,1746.162539,205.549719,778.765766,41.412965,0.505036,595.942674,6.565470,12.120868
3,1400000US06073017039,5.158754e+07,6915,814,3084,164,2,2360,26,48,...,8.801358e+05,0.017061,117.976929,13.887667,52.616175,2.798007,0.034122,40.263999,0.443586,0.818929
4,1400000US06073017043,3.267714e+07,5891,654,2248,161,15,2370,11,31,...,2.889643e+07,0.884301,5209.418868,578.333040,1987.909288,142.372507,13.264519,2095.794044,9.727314,27.413340


In [73]:
# Sum weighted race counts by beat
neighborhoods_race = (
    intersections
    .groupby("NAME")[weighted_cols]
    .sum()
    .reset_index()
)

In [77]:
# Rename columns to remove "_weighted" suffix
neighborhoods_race = neighborhoods_race.rename(columns={
    f"{col}_weighted": col for col in race_cols + ["Total_Pop"]
})

In [78]:
# Determine the predominant race in each neighborhood
neighborhoods_race["predom_race"] = neighborhoods_race[race_cols].idxmax(axis=1)

In [79]:
neighborhoods_race.head()

,NAME,Total_Pop,Hispanic_Latino,White,Black_African_American,American_Indian_Alaska_Native,Asian,Native_Hawaiian_Pacific_Islander,Other_Race,predom_race
0,ADAMS NORTH,4693.048182,1244.567380,2692.095646,203.972437,12.005492,230.186531,10.394593,28.112371,White
1,ALLIED GARDENS,11305.209167,2215.565774,6945.596678,415.950730,39.387174,815.787719,57.755887,48.434528,White
2,ALTA VISTA,2274.665582,546.970512,138.460653,304.468954,8.541510,1152.406733,17.130627,9.447353,Asian
3,AZALEA/HOLLYWOOD PARK,2822.036895,1384.451058,673.786029,288.388955,10.264668,312.103343,9.873470,20.080664,Hispanic_Latino
4,BALBOA PARK,2530.831125,561.455252,1392.877081,179.605745,12.144823,215.548858,5.622929,21.719629,White


In [80]:
# Merge with neighborhood areas
neighborhoods_predom = neighborhoods.merge(neighborhoods_race, on="NAME", how="left")

In [81]:
# Calculate proportion of each race
for col in race_cols:
    neighborhoods_predom[f"prop_{col}"] = neighborhoods_predom[col] / neighborhoods_predom["Total_Pop"]

In [82]:
neighborhoods_predom.head()

,NAME,Shape_Leng,Shape_Area,geometry,neighborhood_area,Total_Pop,Hispanic_Latino,White,Black_African_American,American_Indian_Alaska_Native,...,Native_Hawaiian_Pacific_Islander,Other_Race,predom_race,prop_Hispanic_Latino,prop_White,prop_Black_African_American,prop_American_Indian_Alaska_Native,prop_Asian,prop_Native_Hawaiian_Pacific_Islander,prop_Other_Race
0,NORTH CITY,78873.790699,1.331479e+08,"POLYGON ((6258473.516 1939877.994, 6258489.997...",1.331479e+08,11046.956941,842.003523,5836.888542,79.787916,6.742244,...,8.654127,64.760794,White,0.076220,0.528371,0.007223,0.000610,0.327350,0.000783,0.005862
1,NESTOR,35035.587421,4.154836e+07,"POLYGON ((6302781 1793246.001, 6302905 1793244...",4.154836e+07,13213.206497,10073.943556,1054.959630,443.356976,28.416499,...,47.721659,36.581853,Hispanic_Latino,0.762415,0.079841,0.033554,0.002151,0.090851,0.003612,0.002769
2,BIRDLAND,25427.283539,2.168501e+07,"POLYGON ((6284667.652 1874418.895, 6284694.392...",2.168501e+07,3635.124438,1033.205738,1449.508983,317.766976,26.423661,...,20.169379,23.690673,White,0.284228,0.398751,0.087416,0.007269,0.143112,0.005548,0.006517
3,CHEROKEE POINT,13323.824075,9.452499e+06,"POLYGON ((6297431.051 1853639.085, 6297418.091...",9.452499e+06,4962.370073,2843.526445,1083.098905,364.656403,8.814200,...,9.739972,27.669931,Hispanic_Latino,0.573018,0.218262,0.073484,0.001776,0.090122,0.001963,0.005576
4,LOMA PORTAL,27729.191262,1.889037e+07,"POLYGON ((6263446.993 1854500.089, 6263451.987...",1.889037e+07,5647.453532,921.564038,3807.656177,183.659828,18.881962,...,14.258190,74.051422,White,0.163182,0.674225,0.032521,0.003343,0.041240,0.002525,0.013112


In [ ]:
# tracts_race['prop_hispanic'] = tracts_race['Hispanic_Latino'] / tracts_race['Total_Pop']
# tracts_race['prop_white'] = tracts_race['White'] / tracts_race['Total_Pop']
# tracts_race['prop_black'] = tracts_race['Black_African_American'] / tracts_race['Total_Pop']
# tracts_race['prop_aian'] = tracts_race['American_Indian_Alaska_Native'] / tracts_race['Total_Pop']
# tracts_race['prop_asian'] = tracts_race['Asian'] / tracts_race['Total_Pop']
# tracts_race['prop_nhpi'] = tracts_race['Native_Hawaiian_Pacific_Islander'] / tracts_race['Total_Pop']
# tracts_race['prop_other'] = tracts_race['Other_Race'] / tracts_race['Total_Pop']

# Merging Median Income into SDPD Law Beats (neighborhoods)

In [96]:
# Load Median Income by census tract data
income = gpd.read_file("C:\\UC San Diego\\Capstone\\SimplyAnalytics_income_households\\SimplyAnalytics_Shapefiles_d0142391a3cad5174805042492fd341a86b0375c1d98337f40d57e0914fc708f.shp")

In [97]:
income = income.rename(columns={
    'VALUE0': 'total_pop', 
    'VALUE1': 'older_than_65', 
    'VALUE2': 'bachelor_or_higher', 
    'VALUE3': 'median_income', 
    'VALUE4': 'avg_per_capita_income',
    'VALUE5': 'num_households'})

In [98]:
income.head()

,spatial_id,name,total_pop,older_than_65,bachelor_or_higher,median_income,avg_per_capita_income,num_households,geometry
0,06073000100,"CT000100, San Diego County, CA",2714.0,26.7870,75.6980,221811.27,137196.81,1079.0,"POLYGON ((-117.1949 32.75278, -117.19471 32.75..."
1,06073000201,"CT000201, San Diego County, CA",2476.0,29.8465,72.8550,120824.73,125694.99,1263.0,"POLYGON ((-117.17887 32.75765, -117.17797 32.7..."
2,06073000202,"CT000202, San Diego County, CA",3861.0,16.1875,56.5562,126025.68,90019.81,2241.0,"POLYGON ((-117.18404 32.74571, -117.18383 32.7..."
3,06073000301,"CT000301, San Diego County, CA",2606.0,18.2272,59.6120,88787.38,74450.60,1369.0,"POLYGON ((-117.16864 32.74897, -117.1684 32.74..."
4,06073000302,"CT000302, San Diego County, CA",2903.0,27.9366,60.9795,93720.11,87831.90,1880.0,"POLYGON ((-117.164 32.74091, -117.164 32.74132..."


In [99]:
neighborhoods.head()

,NAME,Shape_Leng,Shape_Area,geometry,neighborhood_area
0,NORTH CITY,78873.790699,1.331479e+08,"POLYGON ((6258473.516 1939877.994, 6258489.997...",1.331479e+08
1,NESTOR,35035.587421,4.154836e+07,"POLYGON ((6302781 1793246.001, 6302905 1793244...",4.154836e+07
2,BIRDLAND,25427.283539,2.168501e+07,"POLYGON ((6284667.652 1874418.895, 6284694.392...",2.168501e+07
3,CHEROKEE POINT,13323.824075,9.452499e+06,"POLYGON ((6297431.051 1853639.085, 6297418.091...",9.452499e+06
4,LOMA PORTAL,27729.191262,1.889037e+07,"POLYGON ((6263446.993 1854500.089, 6263451.987...",1.889037e+07


In [100]:
# Reproject to same CRS
income = income.to_crs(epsg=2230)
neighborhoods = neighborhoods.to_crs(epsg=2230)

In [101]:
# Calculate tract area
income["tract_area"] = income.geometry.area

In [102]:
# Perform spatial intersection between income tracts and beats
income_intersections = gpd.overlay(
    income[["spatial_id", "median_income", "num_households", "tract_area", "geometry"]],
    neighborhoods[["NAME", "geometry"]],
    how="intersection"
)

In [104]:
# Calculate area of intersection and weight
income_intersections["intersect_area"] = (
    income_intersections.geometry.area
)

income_intersections["area_weight"] = (
    income_intersections["intersect_area"] /
    income_intersections["tract_area"])

In [105]:
# Estimate houeholds in each intersection
income_intersections["weighted_households"] = (
    income_intersections["num_households"] *
    income_intersections["area_weight"]
)

In [106]:
# Estimate total income in each intersection
income_intersections["income_x_households"] = (
    income_intersections["median_income"] *
    income_intersections["weighted_households"]
)

In [107]:
income_intersections.head()

,spatial_id,median_income,num_households,tract_area,NAME,geometry,intersect_area,area_weight,weighted_households,income_x_households
0,06073000100,221811.27,1079.0,1.653620e+07,OLD TOWN,"MULTIPOLYGON (((6271212.508 1855074.386, 62713...",2.029510e+06,0.122731,132.427159,2.937384e+07
1,06073000100,221811.27,1079.0,1.653620e+07,MISSION HILLS,"MULTIPOLYGON (((6271383.115 1855884.979, 62713...",1.383184e+07,0.836458,902.538695,2.001933e+08
2,06073000100,221811.27,1079.0,1.653620e+07,MISSION VALLEY WEST,"POLYGON ((6273656.52 1857154.361, 6274061.003 ...",6.748444e+05,0.040810,44.034140,9.767269e+06
3,06073000201,120824.73,1263.0,9.302335e+06,MIDTOWN,"MULTIPOLYGON (((6278319.375 1853858.796, 62781...",1.239877e+02,0.000013,0.016834,2.033976e+03
4,06073000201,120824.73,1263.0,9.302335e+06,MISSION HILLS,"POLYGON ((6278375.963 1853860.116, 6278320.809...",7.741379e+06,0.832197,1051.065410,1.269947e+08


In [108]:
# Aggregate by neighborhood
neighborhoods_income = (
    income_intersections
    .groupby("NAME")[["income_x_households", "weighted_households"]]
    .agg({
        "income_x_households": "sum",
        "weighted_households": "sum"
    })
    .reset_index()
)

# Calculate estimated median income per neighborhood
neighborhoods_income["estimated_median_income"] = (
    neighborhoods_income["income_x_households"] /
    neighborhoods_income["weighted_households"]
)

In [110]:
# Join back to neighborhoods
neighborhoods_income = neighborhoods.merge(
    neighborhoods_income,
    on="NAME",
    how="left"
)

In [111]:
neighborhoods_income.head()

,NAME,Shape_Leng,Shape_Area,geometry,neighborhood_area,income_x_households,weighted_households,estimated_median_income
0,NORTH CITY,78873.790699,1.331479e+08,"POLYGON ((6258473.516 1939877.994, 6258489.997...",1.331479e+08,8.384913e+08,3818.519954,219585.429187
1,NESTOR,35035.587421,4.154836e+07,"POLYGON ((6302781 1793246.001, 6302905 1793244...",4.154836e+07,2.350563e+08,3677.374500,63919.606491
2,BIRDLAND,25427.283539,2.168501e+07,"POLYGON ((6284667.652 1874418.895, 6284694.392...",2.168501e+07,1.445912e+08,1268.400702,113994.910807
3,CHEROKEE POINT,13323.824075,9.452499e+06,"POLYGON ((6297431.051 1853639.085, 6297418.091...",9.452499e+06,1.455887e+08,1854.727630,78496.024404
4,LOMA PORTAL,27729.191262,1.889037e+07,"POLYGON ((6263446.993 1854500.089, 6263451.987...",1.889037e+07,2.952416e+08,2287.355017,129075.534917
